# Notebook 3 - Evaluasi & Komparasi + Ekspor TFLite
Evaluasi pada data test (belum pernah dilihat). Karena data imbalanced, **threshold**
keputusan dioptimalkan pada validation set (memaksimalkan F1), lalu diterapkan ke test.

In [ ]:
import numpy as np, tensorflow as tf, json
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix, roc_curve, precision_recall_curve,
    classification_report)
sns.set_style("whitegrid")
d = np.load("../outputs/processed.npz")
Xva,Xte,yva,yte = d["Xva"],d["Xte"],d["yva"],d["yte"]
PAL = {"MLP":"#2563eb","TabNet":"#16a34a"}
models = {n: tf.keras.models.load_model(f"../outputs/models/{n}.keras") for n in ["MLP","TabNet"]}

In [ ]:
def best_thr(m):
    p = m.predict(Xva, verbose=0).ravel()
    pr, rc, th = precision_recall_curve(yva, p)
    f1 = 2*pr*rc/(pr+rc+1e-9); return float(th[np.argmax(f1[:-1])])

results, probs, preds = {}, {}, {}
for n,m in models.items():
    thr = best_thr(m); p = m.predict(Xte, verbose=0).ravel(); yp = (p>=thr).astype(int)
    results[n] = dict(accuracy=accuracy_score(yte,yp), precision=precision_score(yte,yp,zero_division=0),
        recall=recall_score(yte,yp,zero_division=0), f1=f1_score(yte,yp,zero_division=0),
        auc=roc_auc_score(yte,p), mcc=matthews_corrcoef(yte,yp), threshold=thr)
    probs[n]=p; preds[n]=yp
    print(f"\n=== {n} (threshold={thr:.3f}) ===")
    print(classification_report(yte, yp, target_names=["Tdk Berisiko","Berisiko"], zero_division=0))

### Confusion Matrix

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10,4))
for a,n in zip(ax, ["MLP","TabNet"]):
    cm = confusion_matrix(yte, preds[n])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=a,
        xticklabels=["Tdk Berisiko","Berisiko"], yticklabels=["Tdk Berisiko","Berisiko"])
    a.set_title(f"Confusion Matrix - {n}"); a.set_xlabel("Prediksi"); a.set_ylabel("Aktual")
plt.tight_layout(); plt.show()

### Kurva ROC & Precision-Recall

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,5))
for n in results:
    fpr,tpr,_ = roc_curve(yte, probs[n]); ax[0].plot(fpr,tpr,color=PAL[n],label=f"{n} (AUC={results[n]['auc']:.3f})")
    pr,rc,_ = precision_recall_curve(yte, probs[n]); ax[1].plot(rc,pr,color=PAL[n],label=n)
ax[0].plot([0,1],[0,1],"--",color="grey"); ax[0].set_title("ROC"); ax[0].legend()
ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR")
ax[1].axhline(yte.mean(),ls="--",color="grey"); ax[1].set_title("Precision-Recall")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].legend()
plt.tight_layout(); plt.show()

### Tabel Perbandingan

In [ ]:
import pandas as pd
tbl = pd.DataFrame(results).T[["accuracy","precision","recall","f1","auc","mcc"]].round(4)
print(tbl)
best = tbl["f1"].idxmax(); print(f"\nMODEL TERBAIK (F1): {best}")

### Ekspor ke TensorFlow Lite (untuk Flutter)
Model terbaik diekspor ke `.tflite` untuk dijalankan on-device di aplikasi mobile.

In [ ]:
for n,m in models.items():
    conv = tf.lite.TFLiteConverter.from_keras_model(m)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    open(f"../outputs/models/{n}.tflite","wb").write(conv.convert())
    print(f"{n}.tflite tersimpan")

# simpan threshold + model terbaik ke preprocessing.json
prep = json.load(open("../outputs/models/preprocessing.json"))
prep["thresholds"] = {n: results[n]["threshold"] for n in results}
prep["best_model"] = best
json.dump(prep, open("../outputs/models/preprocessing.json","w"), indent=2)
print("Selesai. Model terbaik:", best)